<a href="https://colab.research.google.com/github/francescoladu/RetrievalSphere/blob/main/RetrievalSphere.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Let's see how this thing works

In [ ]:
%pip install -q \
    ir-datasets==0.6.3 \
    ir-measures==0.4.3 \
    PyStemmer==3.1.0 \
    numpy==2.2.6 \
    pandas==2.2.3 \
    scipy==1.17.1 \
    tqdm==4.70.1 \
    psutil==7.2.2 \
    matplotlib==3.11.2

## Imports and configuration

In [ ]:
import os
import re
import subprocess
import unicodedata
from dataclasses import dataclass
from functools import lru_cache
from pathlib import Path
from typing import Callable

import nltk

try:
    nltk.pathsec.ALLOW_PROXIED_FETCH = True
except AttributeError:
    os.environ["NLTK_ALLOW_PROXIED_URLOPEN"] = "1"

nltk.download("stopwords", quiet=True)

from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer

ACTIVE_LANGUAGES = ("en", "it")
STEM_CACHE_SIZE = 250_000

REMOVE_STOPWORDS = True
APPLY_STEMMING = True

DATA_DIR = Path("/content/retrievalsphere/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

COLLECTION_URL = (
    "https://msmarco.z22.web.core.windows.net/msmarcoranking/collection.tar.gz"
)

## Data

In [ ]:
def fetch_collection(data_dir: Path = DATA_DIR) -> Path:
    """Download and extract the MS MARCO passage collection (idempotent)."""
    data_dir.mkdir(parents=True, exist_ok=True)
    archive = data_dir / "collection.tar.gz"
    collection = data_dir / "collection.tsv"

    if not archive.exists():
        print("Downloading MS MARCO passage collection...")
        subprocess.run(["wget", "-c", COLLECTION_URL, "-O", str(archive)], check=True)

    if not collection.exists():
        print("Extracting collection...")
        subprocess.run(["tar", "-xzf", str(archive), "-C", str(data_dir)], check=True)

    return collection


COLLECTION_PATH = fetch_collection()
print(f"Collection path: {COLLECTION_PATH}")

Extracting collection...
Collection path: /content/retrievalsphere/data/collection.tsv


## Normalization and expansion

In [ ]:
def parse_collection_line(line: str):
    """
    Parse one MS MARCO collection line.

    Expected format:
        <pid>\t<text>

    Returns:
        (pid, text) if the line is valid;
        None otherwise.
    """

    if "\t" not in line:
        return None

    pid_str, text = line.split("\t", 1)

    try:
        pid = int(pid_str)
    except ValueError:
        return None

    text = text.rstrip("\r\n")

    return pid, text

In [ ]:
EXPANSIONS = {
    "en": {
        "exact": {
            "can't": "cannot", "won't": "will not", "shan't": "shall not",
            "don't": "do not", "doesn't": "does not", "didn't": "did not",
            "isn't": "is not", "aren't": "are not", "wasn't": "was not",
            "weren't": "were not", "haven't": "have not", "hasn't": "has not",
            "hadn't": "had not", "couldn't": "could not", "wouldn't": "would not",
            "shouldn't": "should not", "mustn't": "must not",
            "i'm": "i am",
            "you're": "you are", "we're": "we are", "they're": "they are",
            "i've": "i have", "you've": "you have", "we've": "we have", "they've": "they have",
            "i'll": "i will", "you'll": "you will", "we'll": "we will", "they'll": "they will",
            "there's": "there is", "that's": "that is", "what's": "what is",
            "where's": "where is", "who's": "who is",
            "e.g.": "for example", "i.e.": "that is", "etc.": "etcetera",
        },
    },
    "it": {
        "exact": {
            "po'": "poco", "com'è": "come è", "cos'è": "cosa è",
            "ecc.": "eccetera", "dott.": "dottore",
        },
        "prefix": {"un'": "una "},
    },
    "fr": {
        "exact": {"c'est": "ce est", "n'est": "ne est", "etc.": "etcetera"},
        "prefix": {"j'": "je ", "qu'": "que ", "n'": "ne ", "c'": "ce "},
    },
    "es": {
        "exact": {"p.ej.": "por ejemplo", "aprox.": "aproximadamente"},
    },
}

APOSTROPHES = str.maketrans({"’": "'", "‘": "'", "`": "'", "´": "'"})


def normalize_surface(text: str) -> str:
    """Unicode NFKC, apostrophe folding, case folding."""
    return unicodedata.normalize("NFKC", text).translate(APOSTROPHES).casefold()


def build_expander(languages: tuple[str, ...]) -> Callable[[str], str]:
    """Compile the expansion rules of `languages` into a single function."""
    exact: dict[str, str] = {}
    prefix: dict[str, str] = {}
    for language in languages:
        rules = EXPANSIONS.get(language, {})
        exact.update(rules.get("exact", {}))
        prefix.update(rules.get("prefix", {}))

    prefix_rules = [
        (re.compile(rf"(?<!\w){re.escape(p)}(?=\w)"), replacement)
        for p, replacement in prefix.items()
    ]

    keys = sorted(exact, key=len, reverse=True)
    exact_re = (
        re.compile(r"(?<!\w)(?:" + "|".join(map(re.escape, keys)) + r")(?!\w)")
        if keys else None
    )

    def expand(text: str) -> str:
        for pattern, replacement in prefix_rules:
            text = pattern.sub(replacement, text)
        if exact_re:
            text = exact_re.sub(lambda m: exact[m.group(0)], text)
        return text

    return expand


expand_known_forms = build_expander(ACTIVE_LANGUAGES)

## Tokenizer

In [ ]:
_TRAILING_PUNCT = ".,;:!?)]}"


def _identity(token: str) -> str:
    return token


def _clean_url(token: str) -> str:
    token = token.rstrip(_TRAILING_PUNCT)
    token = re.sub(r"^(?:https?://)?(?:www\.)?", "", token)
    return token.rstrip("/")


@dataclass(frozen=True)
class TokenRule:
    name: str
    pattern: str
    canonicalize: Callable[[str], str] = _identity


TOKEN_RULES = (
    TokenRule("url",       r"(?:https?://|www\.)[^\s<>{}\[\]\"']+", _clean_url),
    TokenRule("email",     r"[^\s@]+@[^\s@]+\.[^\s@]+", lambda t: t.rstrip(_TRAILING_PUNCT)),
    TokenRule("date",      r"\b\d{1,4}[/-]\d{1,2}[/-]\d{1,4}\b", lambda t: re.sub(r"[/-]", "-", t)),
    TokenRule("price",     r"[$€£¥]\s?\d+(?:[.,]\d+)*", lambda t: re.sub(r"\s+", "", t)),
    TokenRule("decimal",   r"\b\d+(?:[.,]\d+)+\b"),
    TokenRule("acronym",   r"\b(?:[^\W\d_]{1,4}\.){2,}[^\W\d_]{0,4}\.?", lambda t: t.replace(".", "")),
    TokenRule("ampersand", r"\b[^\W_]+(?:&[^\W_]+)+\b", lambda t: t.replace("&", "")),
    TokenRule("hashtag",   r"#[^\W_]+(?:_[^\W_]+)*", lambda t: t[1:]),
    TokenRule("word",      r"[^\W_]+(?:'[^\W_]+)*"),
)

_RULES_BY_NAME = {rule.name: rule for rule in TOKEN_RULES}
_TOKEN_RE = re.compile("|".join(f"(?P<{r.name}>{r.pattern})" for r in TOKEN_RULES))


def tokenize(text: str) -> list[str]:
    """normalize -> expand known forms -> regex tokenize -> canonicalize."""
    if not text:
        return []

    text = expand_known_forms(normalize_surface(text))
    tokens = (
        _RULES_BY_NAME[m.lastgroup].canonicalize(m.group())
        for m in _TOKEN_RE.finditer(text)
    )
    return [token for token in tokens if token]

## Stop words and stemming

In [ ]:
STOPWORDS = frozenset(stopwords.words("english"))

_stemmer = SnowballStemmer("english")


@lru_cache(maxsize=STEM_CACHE_SIZE)
def stem_word(token: str) -> str:
    return _stemmer.stem(token)


def drop_stopwords(tokens: list[str]) -> list[str]:
    return [t for t in tokens if t not in STOPWORDS]


def stem_tokens(tokens: list[str]) -> list[str]:
    """Stem alphabetic tokens only; URLs, emails, prices, dates, decimals stay as they are."""
    return [stem_word(t) if t.isalpha() else t for t in tokens]


def process_text(
    text: str,
    remove_stopwords: bool | None = None,
    apply_stemming: bool | None = None,
) -> list[str]:
    """Same function for passages and queries. Explicit arguments override the config."""
    if remove_stopwords is None:
        remove_stopwords = REMOVE_STOPWORDS
    if apply_stemming is None:
        apply_stemming = APPLY_STEMMING

    tokens = tokenize(text)
    if remove_stopwords:
        tokens = drop_stopwords(tokens)
    if apply_stemming:
        tokens = stem_tokens(tokens)
    return tokens

## Test

In [ ]:
TOKENIZE_CASES = [
    ("There's no way I can't do it.", ["there", "is", "no", "way", "i", "cannot", "do", "it"]),
    ("I can’t do it.",                ["i", "cannot", "do", "it"]),          # curly apostrophe
    ("Un'amica è arrivata.",          ["una", "amica", "è", "arrivata"]),
    ("Ph.D. and m.p.h.",              ["phd", "and", "mph"]),
    ("AT&T and R&D",                  ["att", "and", "rd"]),
    ("It costs $45.55.",              ["it", "costs", "$45.55"]),
    ("Dates: 01/02/06 and 01-02-06.", ["dates", "01-02-06", "and", "01-02-06"]),
    ("Visit http://www.unipi.it.",    ["visit", "unipi.it"]),
    ("Visit https://www.unipi.it/",   ["visit", "unipi.it"]),
    ("Follow #sigir.",                ["follow", "sigir"]),
    ("Contact name.surname@unipi.it.", ["contact", "name.surname@unipi.it"]),
    ("Caffè, déjà-vu — naïve!",       ["caffè", "déjà", "vu", "naïve"]),
    ("",                              []),
]

PROCESS_CASES = [
    # (text, kwargs, expected)
    ("",                       dict(remove_stopwords=False, apply_stemming=False), []),
    ("The houses are running", dict(remove_stopwords=False, apply_stemming=False), ["the", "houses", "are", "running"]),
    ("The houses are running", dict(remove_stopwords=True,  apply_stemming=False), ["houses", "running"]),
    ("The houses are running", dict(),                                              ["hous", "run"]),
    ("It costs $45.55 at www.unipi.it", dict(),                                    ["cost", "$45.55", "unipi.it"]),
]

for text, expected in TOKENIZE_CASES:
    assert tokenize(text) == expected, f"tokenize({text!r}) -> {tokenize(text)} != {expected}"

for text, kwargs, expected in PROCESS_CASES:
    got = process_text(text, **kwargs)
    assert got == expected, f"process_text({text!r}, {kwargs}) -> {got} != {expected}"

print("All text-processing tests passed.")

All text-processing tests passed.
